In [ ]:
# Install packages
!pip install --upgrade xee
!pip install -U geemap

In [1]:
# Initialize gee
import geemap
import ee
import pandas as pd
import geopandas as gpd

ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Carregar camada
path_json = "/content/drive/MyDrive/desertification/dados/bacias_meso_SF.geojson"
gdf = gpd.read_file(path_json)
geom_ee = geemap.geopandas_to_ee(gdf)
area = geom_ee.geometry()

In [4]:
# 3. Carregar a coleção MODIS (Reflectância de Superfície)
modis = ee.ImageCollection('MODIS/061/MOD09A1') \
    .filterBounds(area) \
    .filterDate('2000-02-18', '2025-12-31')

In [5]:
# 4. Função para calcular MSAVI e Albedo
def add_indices(image):
    # Aplicar fator de escala do MODIS para obter a reflectância real [7]
    img_scaled = image.multiply(0.0001)

    # Cálculo do MSAVI usando .expression() [4]
    msavi = img_scaled.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': img_scaled.select('sur_refl_b02'), # Banda NIR no MODIS
            'RED': img_scaled.select('sur_refl_b01')  # Banda RED no MODIS
        }).rename('MSAVI')

    # Cálculo do Albedo (Exemplo usando fórmula empírica comum para MODIS)
    # Verifique os coeficientes exatos da metodologia que você está seguindo
    albedo = img_scaled.expression(
        '0.160 * B1 + 0.291 * B2 + 0.243 * B3 + 0.116 * B4 + 0.112 * B5 + 0.081 * B7 - 0.0015', {
            'B1': img_scaled.select('sur_refl_b01'),
            'B2': img_scaled.select('sur_refl_b02'),
            'B3': img_scaled.select('sur_refl_b03'),
            'B4': img_scaled.select('sur_refl_b04'),
            'B5': img_scaled.select('sur_refl_b05'),
            'B7': img_scaled.select('sur_refl_b07')
        }).rename('Albedo')

    # Adiciona as novas bandas à imagem original e mantém as propriedades de data [4, 8]
    return image.addBands([msavi, albedo]).copyProperties(image, ['system:time_start'])

# 1. Função para extrair a média e a data de cada imagem
def extrair_serie(image):
    # Calcula a média do MSAVI e Albedo dentro do seu ROI
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area,
        scale=500, # Resolução nativa do MODIS em metros
        maxPixels=1e9
    )

    # Retorna os dados como atributos de uma Feature (sem a geometria pesada)
    return ee.Feature(None, {
        'Data': image.date().format('YYYY-MM-dd'),
        'MSAVI': stats.get('MSAVI'),
        'Albedo': stats.get('Albedo')
    })

In [6]:
# 5. Mapear a função sobre toda a coleção de imagens
modis_com_indices = modis.map(add_indices)

# Testar a performance no XEE

In [7]:
# 1. Inicializar o XEE
import xarray as xr
from xee import helpers
import geopandas as gpd


# Lê o arquivo GeoJSON localmente para o ambiente Python
gdf = gpd.read_file(path_json)

# Extrai a geometria do GeoPandas
aoi = gdf.geometry.union_all()

# Definir parâmetros para fit da geometria
GRID_CRS = "+proj=cea +lon_0=0 +lat_ts=0 +datum=WGS84 +units=m +no_defs"
AOI_CRS = "+proj=longlat +datum=WGS84 +no_defs"
GRID_SCALE = (1000, -1000)

grid_params = helpers.fit_geometry(
    geometry=aoi,
    geometry_crs=AOI_CRS,
    grid_crs=GRID_CRS,
    grid_scale=GRID_SCALE,
)

# Filtrar o a feature collection apenas com as bandas de interesse (MSAVI e Albedo)
modis_com_indices = modis_com_indices.select(["MSAVI", "Albedo"])

# Abre a coleção do GEE como um cubo de dados multidimensional
ds = xr.open_dataset(
    modis_com_indices,
    engine='ee',
    **grid_params,
    chunks={"time": 1, "y": 512, "x": 512},
)
ds

/tmp/ipykernel_20807/1294267440.py:29: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


<xarray.Dataset> Size: 18GB
Dimensions:  (time: 1189, y: 1464, x: 1263)
Coordinates:
  * time     (time) datetime64[ns] 10kB 2000-02-18 2000-02-26 ... 2025-12-27
  * y        (y) float64 12kB -8.015e+05 -8.025e+05 ... -2.264e+06 -2.264e+06
  * x        (x) float64 10kB -5.304e+06 -5.302e+06 ... -4.042e+06 -4.042e+06
Data variables:
    MSAVI    (time, y, x) float32 9GB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    Albedo   (time, y, x) float32 9GB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>

In [8]:
ds=ds.sortby('time')*1

In [9]:


import numpy as np
# ---------------------------------------------------------
# ABORDAGEM 1: DDI (Desertification Divided Index) - Regressão Linear
# Utilizando o coeficiente empírico (K = 1.803) validado na literatura [11]
# Valores MAIORES indicam áreas severamente preservadas (ou não-desertificadas, dependendo do sinal do índice).
# A literatura em [11] frequentemente inverte o eixo para mapear Intensidade I.
# No modelo padrão DDI (K * MSAVI - Albedo), o decréscimo representa degradação [10].
# ---------------------------------------------------------
K_coef = 1.803
ds['DDI'] = (K_coef * ds['MSAVI']) - ds['Albedo']

# ---------------------------------------------------------
# ABORDAGEM 2: SASDI (Point-to-Point Distance Model)
# Calcula a distância euclidiana do pixel em relação ao estado ideal (MSAVI=1, Albedo=0) [12, 13]
# Quanto MAIOR a distância, MAIOR o grau de desertificação.
# O xarray aplica a operação vetorial elemento a elemento perfeitamente no Dask Graph.
# ---------------------------------------------------------
ds['SASDI'] = np.sqrt((ds['MSAVI'] - 1)**2 + (ds['Albedo'])**2)

In [16]:
# Para visualização de mapas, vamos reduzir a resolução (coarsen)
# Isso agrupa blocos de pixels (ex: 10x10) e tira a média, diminuindo o volume de dados
ds_light = ds.coarsen(x=10, y=10, boundary='trim').mean()

# Agora agrupamos por ano (mediana anual)
ds_year = ds_light.resample(time="YE").median().compute()

# Plotando os mapas resultantes
ds_year['DDI'].plot(x='x', y='y', col='time', col_wrap=3, robust=True, cmap='Reds')

EEException: Could not parse '+proj=cea +lon_0=0 +lat_ts=0 +datum=WGS84 +units=m +no_defs'.

In [11]:
# 2. Prepara o cálculo da média espacial do MSAVI, Albedo, DDI e SASDI
# O Dask organiza as requisições para rodarem em paralelo no servidor do Google
serie_media = ds[['MSAVI', 'Albedo', 'DDI', 'SASDI']].mean(dim=['x', 'y'])
serie_media = ds.sel()

### Visualização da Série Temporal dos Índices de Desertificação

Vamos plotar as séries temporais do DDI e SASDI para observar as tendências ao longo do tempo.